In [7]:
import os
import pickle

### Settings

In [8]:
DIR_DATA = "data"
DIR_TEXTS = os.path.join(DIR_DATA, "texts")
FILE_METADATA = os.path.join(DIR_DATA, "metadata.csv")

cleaned_corpus_indices_path = os.path.join(DIR_DATA, "cleaned_corpus_indices.pkl")
cleaned_corpus_token_to_index_path = os.path.join(DIR_DATA, "token_to_index.pkl")
cleaned_corpus_index_to_token_path = os.path.join(DIR_DATA, "index_to_token.pkl")

In [9]:
# Load the cleaned corpus indices and the index-to-token mapping
with open(cleaned_corpus_indices_path, "rb") as f:
    tokens = pickle.load(f)
    
with open(cleaned_corpus_index_to_token_path, "rb") as f:
    index_to_token = pickle.load(f)
    
print(f"Loaded {len(tokens)} tokens and {len(index_to_token)} unique tokens.")

Loaded 12207411 tokens and 46724 unique tokens.


### Bi-gram

In [10]:
PROGRESS_RESOLUTION = 50000
PROGRESS_BAR_WIDTH = 50
NS = [2, 3, 4, 5]

def get_ngram_counts(input_tokens: list[int], ns: list[int]):

    # Allows us to safely start the loop at 1
    assert all(n >= 2 for n in ns), "n-grams only make sense for n >= 2"
    
    # Big dictionary of dictionaries
    ngram_counts = {
        n: {} for n in ns
    }
    
    # Iterate through each token in the corpus (skip first n-1 tokens)
    for token_i in range(1, len(input_tokens)):
        
        # Print progress every PROGRESS_RESOLUTION tokens
        if token_i % PROGRESS_RESOLUTION == 0:
            progress = token_i / len(input_tokens)
            bar = "#" * int(progress * PROGRESS_BAR_WIDTH)
            print(f"[{bar:<{PROGRESS_BAR_WIDTH}}] {progress:.2%} ({token_i}/{len(input_tokens)})", end="\r")
            
        for n in ns:
            
            # Skip if not enough tokens past
            if token_i < n - 1:
                continue
            
            # Extract the n-gram
            n_gram = tuple(input_tokens[token_i - n + 1: token_i + 1])
            ngram_counts[n][n_gram] = ngram_counts[n].get(n_gram, 0) + 1

    print(f"[{'#' * PROGRESS_BAR_WIDTH}] 100.00% ({len(input_tokens)}/{len(input_tokens)})")
    return ngram_counts

# Use encoded tokens
ngrams = get_ngram_counts(tokens, NS)
NGRAM_SOURCE = "idx"

# Pickle the n-grams
# We store the mapping so we can reconstruct the strings later
file_name = f"ngrams_{'-'.join(map(str, NS))}_{NGRAM_SOURCE}.pkl"
print("Saving pickle...")
with open(os.path.join(DIR_DATA, file_name), "wb") as f:
    pickle.dump(ngrams, f)
print(f"Saved to {os.path.join(DIR_DATA, file_name)}")

[##################################################] 100.00% (12207411/12207411)
Saving pickle...
Saved to data\ngrams_2-3-4-5_idx.pkl


In [11]:
print("Number of unique n-grams:")
for n in NS:
    print(f"  {n}-grams: {len(ngrams[n])} (out of {len(tokens)**n:.2E} possible n-grams, 1 in {len(tokens)**n / len(ngrams[n]):.2E})")

Number of unique n-grams:
  2-grams: 1615643 (out of 1.49E+14 possible n-grams, 1 in 9.22E+07)
  3-grams: 6083536 (out of 1.82E+21 possible n-grams, 1 in 2.99E+14)
  4-grams: 9905883 (out of 2.22E+28 possible n-grams, 1 in 2.24E+21)
  5-grams: 11373132 (out of 2.71E+35 possible n-grams, 1 in 2.38E+28)


In [14]:
# Show the most common n-grams
print("\nExample n-grams:")
for n in NS:
    print(f"  {n}-grams:")
    most_common = sorted(ngrams[n].items(), key=lambda x: x[1], reverse=True)[:5]
    for n_gram, count in most_common:
        n_gram_str = " ".join(index_to_token[idx] for idx in n_gram)
        print(f"    '{n_gram_str}': {count} occurrences")


Example n-grams:
  2-grams:
    'of the': 86534 occurrences
    'in the': 62280 occurrences
    'to the': 41919 occurrences
    '. i': 40771 occurrences
    '. the': 39451 occurrences
  3-grams:
    '. he was': 6112 occurrences
    '. it was': 5836 occurrences
    'of the and': 4629 occurrences
    '. it is': 3660 occurrences
    'one of the': 3629 occurrences
  4-grams:
    'was killed in action': 1829 occurrences
    'and was killed in': 1659 occurrences
    'in action in the': 1551 occurrences
    'no . battn .': 1483 occurrences
    'lost in action in': 1428 occurrences
  5-grams:
    'and was killed in action': 1521 occurrences
    'in action in the north': 1427 occurrences
    'lost in action in the': 1426 occurrences
    '. lost in action in': 1419 occurrences
    'in the north sept .': 1385 occurrences
